In [2]:
import pandas as pd 
import duckdb 
import logging 

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.metrics import mean_absolute_error

logging.basicConfig(
    level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s',
    filename='analysis.log'
)
logger = logging.getLogger(__name__)

In [3]:
### load duckdb tables into pandas dfs 
con = None
try: 
    # create and verify connection 
    con = duckdb.connect(database='project1.db', read_only=False) 
    logger.info("Connected to duckdb instance.")

    # inserting tables 
    cville = con.execute(f"""
        SELECT * FROM cville;
    """).fetchdf()
    dulles = con.execute(f"""
        SELECT * FROM dulles;
    """).fetchdf()
    lynchburg = con.execute(f"""
        SELECT * FROM lynchburg;
    """).fetchdf()
    norfolk = con.execute(f"""
        SELECT * FROM norfolk;
    """).fetchdf()
    dom = con.execute(f"""
        SELECT * FROM dom;
    """).fetchdf()

    logger.info("All tables loaded into pandas dataframes")

except Exception as e:
    logger.error(f"An error occurred: {e}")

Before starting the analysis, the 4 weather datasets and energy datasets were merged for easier analysis. They were merged on the date columns, and then the `Datetime` column from the DOM_hourly.csv was dropped as to not have repetitive columns.

In [4]:
### combine weather data into one df
weatherdf = pd.concat([cville, dulles, lynchburg, norfolk], ignore_index=True)

# ensure both Date columns are in proper format before merging 
weatherdf['DATE'] = pd.to_datetime(weatherdf['DATE'])
dom['Datetime'] = pd.to_datetime(dom['Datetime'])
# round to the hour to prevent errors
dom['Datetime'] = dom['Datetime'].dt.floor('h')

# merge weather and energy dfs 
combined_df = pd.merge(weatherdf, dom, left_on='DATE', right_on='Datetime', how='inner')
combined_df = combined_df.drop(columns=['Datetime'])
logger.info("Weather and energy dataframes merged into combined_df.")
combined_df  

,STATION,DATE,Sunrise,Sunset,DailyAverageDewPointTemperature,DailyAverageDryBulbTemperature,DailyAverageRelativeHumidity,DailyAverageSeaLevelPressure,DailyAverageStationPressure,DailyAverageWetBulbTemperature,...,DailySnowDepth,DailySnowfall,DailySustainedWindDirection,DailySustainedWindSpeed,DailyWeather,Month,DayOfWeek,DayOfYear,EWE,DOM_MW
0,Charlottesville,2005-12-31 23:00:00,730.0,1704.0,0.0,45.0,0.0,0.00,29.10,0.0,...,0.0,0.0,250.0,17.0,NaN,12,5,365,1.0,9991.0
1,Charlottesville,2006-01-01 23:00:00,730.0,1705.0,0.0,42.0,0.0,0.00,29.36,0.0,...,0.0,0.0,240.0,12.0,NaN,1,6,1,1.0,10467.0
2,Charlottesville,2006-01-02 23:00:00,731.0,1705.0,0.0,35.0,0.0,0.00,29.28,0.0,...,0.0,0.0,40.0,8.0,RA FG BR,1,0,2,1.0,9914.0
3,Charlottesville,2006-01-03 23:00:00,731.0,1706.0,0.0,44.0,0.0,0.00,29.11,0.0,...,0.0,0.0,20.0,7.0,FG BR,1,1,3,1.0,10716.0
4,Charlottesville,2006-01-04 23:00:00,731.0,1707.0,0.0,42.0,0.0,0.00,29.25,0.0,...,0.0,0.0,200.0,12.0,BR HZ,1,2,4,1.0,10798.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
18362,Norfolk,2018-07-29 23:00:00,507.0,1914.0,72.0,81.0,76.0,30.08,30.03,75.0,...,0.0,0.0,70.0,14.0,NaN,7,6,210,0.0,12448.0
18363,Norfolk,2018-07-30 23:00:00,508.0,1914.0,74.0,78.0,92.0,30.06,30.03,75.0,...,0.0,0.0,70.0,17.0,RA FG BR,7,0,211,1.0,11452.0
18364,Norfolk,2018-07-31 23:00:00,509.0,1913.0,75.0,81.0,82.0,30.06,30.02,77.0,...,0.0,0.0,200.0,18.0,NaN,7,1,212,0.0,12787.0
18365,Norfolk,2018-08-01 23:00:00,510.0,1912.0,75.0,84.0,80.0,30.09,30.04,77.0,...,0.0,0.0,190.0,26.0,RA,8,2,213,0.0,13569.0


In [5]:
# save combined_df as parquet for later use
combined_df.to_parquet("data/combined_df.parquet", engine='pyarrow', index=False)
logger.info("combined_df saved as parquet")

# save combined_df to duckdb table for later press release use 
con = None
try: 
    # create and verify connection 
    con = duckdb.connect(database='project1.db', read_only=False) 
    logger.info("Connected to duckdb instance.") 

    # inserting table 
    con.execute(f"""
        DROP TABLE IF EXISTS combined_df;
        CREATE TABLE combined_df AS
        SELECT * FROM read_parquet('data/combined_df.parquet');
    """)

    logger.info("combined_df loaded into duckdb table.")

except Exception as e:
    logger.error(f"An error occurred: {e}")

For the `Station` column, one-hot encoding was used so that the different cities could be added as a feature for analysis. When forming the X dataframe, unnecessary columns were dropped such as: `Date`, because using other more specific date columns; `DOM_MW`, because is a target variable; `Sunrise/Sunset`, because are not as relevant to the analysis; and `DailyWeather`, because the `EWE` column already accounts for this. 

A train-test split with 80/20 split was used, with the target variable being `DOM_MW`, aka the energy usage. 

In [ ]:
# one-hot encoding 
df = pd.get_dummies(combined_df, columns=['STATION'], drop_first=False)

# dropping columns
X = df.drop(columns=['DATE', 'DOM_MW', 'Sunrise', 'Sunset', 'DailyWeather'])
# target variable is DOM_MW
y = df['DOM_MW']

# Split data into train and test sets
X_train, X_test, y_train, y_test, extreme_train, extreme_test = train_test_split(
    X, y, df['EWE'], test_size=0.2, random_state=42)

A linear regression model and random forest model were used to predict energy usage based on the weather data. The goal with these two models was to have a simpler and them more complex prediction models to see if there were significant differences in the performance metrics. For each model, root mean square error (RMSE), mean absolute error (MAE), and r-sqaured (R^2) were calculated for the performance overall, performance on normal weather conditions, and performance on the extreme weather conditions. Using these three metrics shows the error overall (MAE), the error with large errors being penalized more (RMSE), and the variance in the model (R^2) The results were combined into a pandas dataframe and stored in a duckdb table for later use during the visualization stage.

## Linear Regression Model

In [8]:
### linear regression model
lr = LinearRegression()
lr.fit(X_train, y_train)
y_pred_lr = lr.predict(X_test)

In [9]:
# Overall performance for lr model 
lr_overall_rmse = round(mean_squared_error(y_test, y_pred_lr) ** 0.5, 4)
lr_overall_mae = round(mean_absolute_error(y_test, y_pred_lr), 4)
lr_overall_r2 = round(r2_score(y_test, y_pred_lr), 4)

# Performance by weather type
normal_mask = extreme_test == 0
extreme_mask = extreme_test == 1

# Normal weather
lr_n_rmse = round(mean_squared_error(y_test[normal_mask], y_pred_lr[normal_mask]) ** 0.5, 4)
lr_n_mae = round(mean_absolute_error(y_test[normal_mask], y_pred_lr[normal_mask]), 4)
lr_n_r2 = round(r2_score(y_test[normal_mask], y_pred_lr[normal_mask]), 4)

# Extreme weather
lr_e_rmse = round(mean_squared_error(y_test[extreme_mask], y_pred_lr[extreme_mask]) ** 0.5, 4)
lr_e_mae = round(mean_absolute_error(y_test[extreme_mask], y_pred_lr[extreme_mask]), 4)
lr_e_r2 = round(r2_score(y_test[extreme_mask], y_pred_lr[extreme_mask]), 4)

# put into a dictionary for easy addition to df later 
lr_results = {
    'Model': 'Linear_Regression',
    'Overall_RMSE': lr_overall_rmse,
    'Normal_RMSE': lr_n_rmse,
    'Extreme_RMSE': lr_e_rmse,
    'Overall_MAE': lr_overall_mae,
    'Normal_MAE': lr_n_mae,
    'Extreme_MAE': lr_e_mae,
    'Overall_R2': lr_overall_r2,
    'Normal_R2': lr_n_r2, 
    'Extreme_R2': lr_e_r2}

logger.info('Linear regression model done.')

## Random Forest Model

In [10]:
# training random forest model, with 100 trees 
rf = RandomForestRegressor(n_estimators=100, random_state=42)
rf.fit(X_train, y_train)

# predict on the test set
y_pred = rf.predict(X_test)

In [11]:
# Overall performance
rf_overall_rmse = round(mean_squared_error(y_test, y_pred) ** 0.5, 4)
rf_overall_mae = round(mean_absolute_error(y_test, y_pred), 4)
rf_overall_r2 = round(r2_score(y_test, y_pred), 4)

# Performance by weather type
normal_mask = extreme_test == 0
extreme_mask = extreme_test == 1

# Normal weather
rf_n_rmse = round(mean_squared_error(y_test[normal_mask], y_pred[normal_mask]) ** 0.5, 4)
rf_n_mae = round(mean_absolute_error(y_test[normal_mask], y_pred[normal_mask]), 4)
rf_n_r2 = round(r2_score(y_test[normal_mask], y_pred[normal_mask]), 4)

# Extreme weather
rf_e_rmse = round(mean_squared_error(y_test[extreme_mask], y_pred[extreme_mask]) ** 0.5, 4)
rf_e_mae = round(mean_absolute_error(y_test[extreme_mask], y_pred[extreme_mask]), 4)
rf_e_r2 = round(r2_score(y_test[extreme_mask], y_pred[extreme_mask]), 4)

# put into a dictionary for easy addition to df later
rf_results = {
    'Model': 'Random_Forest',
    'Overall_RMSE': rf_overall_rmse,
    'Normal_RMSE': rf_n_rmse,
    'Extreme_RMSE': rf_e_rmse,
    'Overall_MAE': rf_overall_mae,
    'Normal_MAE': rf_n_mae,
    'Extreme_MAE': rf_e_mae,
    'Overall_R2': rf_overall_r2,    
    'Normal_R2': rf_n_r2,
    'Extreme_R2': rf_e_r2} 

logger.info('Random forest model done.')

The results from the Linear Regression and Random Forest models are combined into one pandas dataframe and saved as a parquet file. The results dataframe is also uploaded to duckdb to be used later for visualizations. 

In [12]:
# create results dataframe to compare models
results_df = pd.DataFrame([lr_results, rf_results])

# save results_df to parquet file for later use
results_df.to_parquet('data/model_results.parquet', engine='pyarrow', index=False)
results_df

,Model,Overall_RMSE,Normal_RMSE,Extreme_RMSE,Overall_MAE,Normal_MAE,Extreme_MAE,Overall_R2,Normal_R2,Extreme_R2
0,Linear_Regression,890.8464,868.4940,911.4645,699.2240,687.3566,710.4381,0.7542,0.6968,0.7818
1,Random_Forest,547.0304,524.8446,567.1979,395.3623,377.9069,411.8567,0.9073,0.8893,0.9155


In [ ]:
# save resultsdf to duckdb table 
con = None
try: 
    # create and verify connection 
    con = duckdb.connect(database='project1.db', read_only=False) 
    logger.info("Connected to duckdb instance.") 

    # inserting table 
    con.execute(f"""
        DROP TABLE IF EXISTS results_df;
        CREATE TABLE results_df AS
        SELECT * FROM read_parquet('data/model_results.parquet');
    """)

    logger.info("results_df loaded into duckdb table.")

except Exception as e:
    logger.error(f"An error occurred: {e}")

finally:
    if con:
        con.close()
        logger.info("Duckdb connection closed.")